<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l8.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L8 · Ética y kill-switch
45 días con freno en -3%: activaciones, drawdown máximo y costo de la vuelta.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l8.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l8.csv'), Path('data/c6_l8.csv'), Path('c6_l8.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['equity'] = 100 * (1 + df['pnl_diario_pct'] / 100).cumprod()
df['peak'] = df['equity'].cummax()
df['dd_check'] = (df['equity'] / df['peak'] - 1) * 100
kills = df[df.kill_switch == 1]
print(int(len(kills)), 'activaciones')
print(kills[['dia', 'pnl_diario_pct']].to_string(index=False))
print('drawdown maximo: %.2f%%' % df['dd_check'].min())

In [ ]:
rec_con = 1 / (1 + df['dd_check'].min() / 100) - 1
sin_freno = df['pnl_diario_pct'].copy()
sin_freno[df.kill_switch == 1] -= 2.0
eq2 = 100 * (1 + sin_freno / 100).cumprod()
dd2 = float((eq2 / eq2.cummax() - 1).min() * 100)
print('recuperacion con freno: +%.1f%%' % (rec_con * 100))
print('drawdown sin freno simulado: %.2f%%' % dd2)

In [ ]:
assert len(df) == 45 and set(['pnl_diario_pct', 'dd_acum_pct', 'kill_switch']) <= set(df.columns)
assert int(df.kill_switch.sum()) == 3
assert abs(df['dd_acum_pct'].min() + 9.95) < 0.05
assert ((df.pnl_diario_pct <= -3.0) == (df.kill_switch == 1)).all()
print('OK L8: 3 frenadas verificadas, drawdown -10,39% contenido')